In [8]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.ensemble import RandomForestRegressor, VotingRegressor, BaggingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import KFold, cross_val_score, train_test_split
import lightgbm as lgb
import optuna

In [9]:
X_train = pd.read_csv('X_train.csv')
X_test  = pd.read_csv('X_test.csv')
y       = pd.read_csv('y_train.csv').squeeze()
ids     = pd.read_csv('test_ids.csv').squeeze()

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y:       {y.shape}")

X_train: (1458, 203)
X_test:  (1459, 203)
y:       (1458,)


In [10]:
# Разбиваем на train/val
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y, test_size=0.2, random_state=42
)

In [11]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Ridge':        Ridge(alpha=10),
    'Lasso':        Lasso(alpha=0.001),
    'ElasticNet':   ElasticNet(alpha=0.001, l1_ratio=0.5),
    'RandomForest': RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42),
    'LightGBM':     lgb.LGBMRegressor(n_estimators=500, learning_rate=0.01,
                                       num_leaves=15, random_state=42, verbose=-1),
    'XGBoost':      XGBRegressor(n_estimators=500, learning_rate=0.01,
                                  max_depth=3, random_state=42, verbosity=0),
    'CatBoost':     CatBoostRegressor(n_estimators=500, learning_rate=0.01,
                                       depth=3, random_seed=42, verbose=0),
    'Voting':       VotingRegressor([
                        ('ElasticNet', ElasticNet(alpha=0.001, l1_ratio=0.5)),
                        ('Ridge',      Ridge(alpha=10)),
                        ('LightGBM',   lgb.LGBMRegressor(n_estimators=500, learning_rate=0.01,
                                                          num_leaves=15, random_state=42, verbose=-1)),
                    ]),
    'Bagging':      BaggingRegressor(
                        estimator=DecisionTreeRegressor(max_depth=5),
                        n_estimators=100,
                        random_state=42
                    ),
}

results = {}
for name, model in models.items():
    cv_scores = cross_val_score(model, X_train, y,
                                cv=cv, scoring='neg_root_mean_squared_error')
    rmse = -cv_scores.mean()
    std  = cv_scores.std()
    results[name] = {'RMSE': round(rmse, 4), 'STD': round(std, 4)}
    print(f"{name:15s}: RMSE={rmse:.4f} ± {std:.4f}")

results_df = pd.DataFrame(results).T.sort_values('RMSE')
results_df

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    alpha   = trial.suggest_float('alpha', 1e-4, 1.0, log=True)
    l1_ratio = trial.suggest_float('l1_ratio', 0.0, 1.0)
    
    model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=10000)
    
    cv_scores = cross_val_score(model, X_train, y,
                                cv=cv, scoring='neg_root_mean_squared_error')
    return -cv_scores.mean()

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=100)

print(f"Лучший RMSE: {study.best_value:.4f}")
print(f"Лучшие параметры: {study.best_params}")

In [ ]:
import matplotlib.pyplot as plt

lgb_model = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.01,
                               num_leaves=15, random_state=42, verbose=-1)
lgb_model.fit(X_train, y)

importance = pd.DataFrame({
    'feature':    X_train.columns,
    'importance': lgb_model.feature_importances_
}).sort_values('importance', ascending=False)

# Топ 30
print(importance.head(30))

# Сколько признаков с нулевой важностью
print(f"\nПризнаков с importance = 0: {(importance['importance'] == 0).sum()}")

In [ ]:
print(f"Признаков с importance = 0: {(importance['importance'] == 0).sum()}")
print(f"Признаков с importance < 10: {(importance['importance'] < 10).sum()}")

# Визуализация
plt.figure(figsize=(10, 6))
plt.hist(importance['importance'], bins=50)
plt.xlabel('Importance')
plt.ylabel('Количество признаков')
plt.title('Распределение важности признаков')
plt.show()

In [ ]:
# Берём только признаки с importance > 0
useful_features = importance[importance['importance'] > 0]['feature'].tolist()
print(f"Оставляем признаков: {len(useful_features)}")

X_train_filtered = X_train[useful_features]
X_test_filtered  = X_test[useful_features]

# Прогоняем все модели заново
results_filtered = {}
for name, model in models.items():
    cv_scores = cross_val_score(model, X_train_filtered, y,
                                cv=cv, scoring='neg_root_mean_squared_error')
    rmse = -cv_scores.mean()
    std  = cv_scores.std()
    results_filtered[name] = {'RMSE': round(rmse, 4), 'STD': round(std, 4)}
    print(f"{name:15s}: RMSE={rmse:.4f} ± {std:.4f}")

pd.DataFrame(results_filtered).T.sort_values('RMSE')

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y, test_size=0.2, random_state=42
)

cb_model = CatBoostRegressor(eval_metric='RMSE')

cb_model.fit(
    X_tr, y_tr,
    eval_set=(X_val, y_val),
)

y_pred_cb = cb_model.predict(X_val)
rmse_cb = np.sqrt(mean_squared_error(y_val, y_pred_cb))
print(f"\nRMSE CatBoost: {rmse_cb:.4f}")

In [12]:
def objective_ridge(trial):
    alpha = trial.suggest_float('alpha', 0.1, 100, log=True)
    
    model = Ridge(alpha=alpha)
    cv_scores = cross_val_score(model, X_train, y,
                                cv=cv, scoring='neg_root_mean_squared_error')
    return -cv_scores.mean()

study_ridge = optuna.create_study(direction='minimize')
study_ridge.optimize(objective_ridge, n_trials=100)

print(f"Лучший RMSE Ridge: {study_ridge.best_value:.4f}")
print(f"Лучшие параметры: {study_ridge.best_params}")

[I 2026-03-26 18:41:34,430] A new study created in memory with name: no-name-d14f7998-508f-4e79-b4b7-2323693817de
[I 2026-03-26 18:41:34,488] Trial 0 finished with value: 0.11142439881308017 and parameters: {'alpha': 19.464500399304857}. Best is trial 0 with value: 0.11142439881308017.
[I 2026-03-26 18:41:34,525] Trial 1 finished with value: 0.11405226409075928 and parameters: {'alpha': 0.760887617837184}. Best is trial 0 with value: 0.11142439881308017.
[I 2026-03-26 18:41:34,563] Trial 2 finished with value: 0.11114022110451396 and parameters: {'alpha': 7.399023500033039}. Best is trial 2 with value: 0.11114022110451396.
[I 2026-03-26 18:41:34,599] Trial 3 finished with value: 0.11478206090216111 and parameters: {'alpha': 0.507674695545711}. Best is trial 2 with value: 0.11114022110451396.
[I 2026-03-26 18:41:34,635] Trial 4 finished with value: 0.11176602073317685 and parameters: {'alpha': 3.307717623825553}. Best is trial 2 with value: 0.11114022110451396.
[I 2026-03-26 18:41:34,67

Лучший RMSE Ridge: 0.1111
Лучшие параметры: {'alpha': 10.071183950181904}


In [13]:
def objective_lgb(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 100, 2000),
        'learning_rate':     trial.suggest_float('learning_rate', 0.001, 0.1, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 10, 50),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'random_state':      42,
        'verbose':           -1,
    }
    
    model = lgb.LGBMRegressor(**params)
    cv_scores = cross_val_score(model, X_train, y,
                                cv=cv, scoring='neg_root_mean_squared_error')
    return -cv_scores.mean()

study_lgb = optuna.create_study(direction='minimize')
study_lgb.optimize(objective_lgb, n_trials=50)

print(f"Лучший RMSE LightGBM: {study_lgb.best_value:.4f}")
print(f"Лучшие параметры: {study_lgb.best_params}")

[I 2026-03-26 18:41:57,340] A new study created in memory with name: no-name-cfc4ba2e-f2fa-41fe-9bd8-fe9341bbd908
[I 2026-03-26 18:42:06,502] Trial 0 finished with value: 0.1297377082215358 and parameters: {'n_estimators': 452, 'learning_rate': 0.008535940080917601, 'num_leaves': 34, 'min_child_samples': 36, 'subsample': 0.9600421346008242, 'colsample_bytree': 0.8092356368956903}. Best is trial 0 with value: 0.1297377082215358.
[I 2026-03-26 18:42:12,807] Trial 1 finished with value: 0.1227928233352585 and parameters: {'n_estimators': 600, 'learning_rate': 0.03956069249872351, 'num_leaves': 11, 'min_child_samples': 41, 'subsample': 0.7959302382974783, 'colsample_bytree': 0.9281498449161455}. Best is trial 1 with value: 0.1227928233352585.
[I 2026-03-26 18:42:46,082] Trial 2 finished with value: 0.15347402963989049 and parameters: {'n_estimators': 1536, 'learning_rate': 0.0012501512614022717, 'num_leaves': 50, 'min_child_samples': 35, 'subsample': 0.8607584456751487, 'colsample_bytree':

Лучший RMSE LightGBM: 0.1169
Лучшие параметры: {'n_estimators': 1812, 'learning_rate': 0.01185060865103241, 'num_leaves': 14, 'min_child_samples': 5, 'subsample': 0.9927865856597863, 'colsample_bytree': 0.7009144370133}


In [14]:
def objective_lasso(trial):
    alpha = trial.suggest_float('alpha', 1e-5, 1.0, log=True)
    
    model = Lasso(alpha=alpha, max_iter=10000)
    cv_scores = cross_val_score(model, X_train, y,
                                cv=cv, scoring='neg_root_mean_squared_error')
    return -cv_scores.mean()

study_lasso = optuna.create_study(direction='minimize')
study_lasso.optimize(objective_lasso, n_trials=100)

print(f"Лучший RMSE Lasso: {study_lasso.best_value:.4f}")
print(f"Лучшие параметры: {study_lasso.best_params}")

[I 2026-03-26 18:58:04,745] A new study created in memory with name: no-name-6fc2224f-01ec-4eb9-9d0e-e287292502d3
[I 2026-03-26 18:58:05,118] Trial 0 finished with value: 0.1299962500541126 and parameters: {'alpha': 0.006363718551020526}. Best is trial 0 with value: 0.1299962500541126.
[I 2026-03-26 18:58:05,469] Trial 1 finished with value: 0.12666836108814272 and parameters: {'alpha': 0.004798211763487051}. Best is trial 1 with value: 0.12666836108814272.
[I 2026-03-26 18:58:05,772] Trial 2 finished with value: 0.12909333624187008 and parameters: {'alpha': 0.005822707887939414}. Best is trial 1 with value: 0.12666836108814272.
[I 2026-03-26 18:58:06,488] Trial 3 finished with value: 0.11890014971192106 and parameters: {'alpha': 0.0024049874038049603}. Best is trial 3 with value: 0.11890014971192106.
[I 2026-03-26 18:58:06,618] Trial 4 finished with value: 0.1751694965211013 and parameters: {'alpha': 0.09809807551075123}. Best is trial 3 with value: 0.11890014971192106.
/Users/beeek18

Лучший RMSE Lasso: 0.1095
Лучшие параметры: {'alpha': 0.0004672260332236319}


In [15]:
# ElasticNet
def objective_elastic(trial):
    alpha    = trial.suggest_float('alpha', 1e-5, 1.0, log=True)
    l1_ratio = trial.suggest_float('l1_ratio', 0.0, 1.0)
    
    model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=10000)
    cv_scores = cross_val_score(model, X_train, y,
                                cv=cv, scoring='neg_root_mean_squared_error')
    return -cv_scores.mean()

study_elastic = optuna.create_study(direction='minimize')
study_elastic.optimize(objective_elastic, n_trials=100)

print(f"Лучший RMSE ElasticNet: {study_elastic.best_value:.4f}")
print(f"Лучшие параметры: {study_elastic.best_params}")

[I 2026-03-26 19:04:13,600] A new study created in memory with name: no-name-142e7bbb-00dc-4f06-adce-6f9224556860
[I 2026-03-26 19:04:13,784] Trial 0 finished with value: 0.17488547071143917 and parameters: {'alpha': 0.12259991393054993, 'l1_ratio': 0.7860869487157024}. Best is trial 0 with value: 0.17488547071143917.
/Users/beeek18/Desktop/ml_project/.venv/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.652e+00, tolerance: 1.836e-02
  model = cd_fast.enet_coordinate_descent(
/Users/beeek18/Desktop/ml_project/.venv/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap:

Лучший RMSE ElasticNet: 0.1095
Лучшие параметры: {'alpha': 0.000467493577952073, 'l1_ratio': 0.9930586023710615}


In [16]:
# Ridge
def objective_ridge(trial):
    alpha = trial.suggest_float('alpha', 0.01, 1000, log=True)
    
    model = Ridge(alpha=alpha)
    cv_scores = cross_val_score(model, X_train, y,
                                cv=cv, scoring='neg_root_mean_squared_error')
    return -cv_scores.mean()

study_ridge = optuna.create_study(direction='minimize')
study_ridge.optimize(objective_ridge, n_trials=100)

print(f"Лучший RMSE Ridge: {study_ridge.best_value:.4f}")
print(f"Лучшие параметры: {study_ridge.best_params}")

[I 2026-03-26 19:11:19,663] A new study created in memory with name: no-name-d04b8a9b-7fe1-4257-b225-f1feb3c8701c
[I 2026-03-26 19:11:19,710] Trial 0 finished with value: 0.11137198805877109 and parameters: {'alpha': 4.978171135303906}. Best is trial 0 with value: 0.11137198805877109.
[I 2026-03-26 19:11:19,751] Trial 1 finished with value: 0.11682723691552721 and parameters: {'alpha': 0.14013981703082515}. Best is trial 0 with value: 0.11137198805877109.
[I 2026-03-26 19:11:19,811] Trial 2 finished with value: 0.11704628471499386 and parameters: {'alpha': 118.7533099900302}. Best is trial 0 with value: 0.11137198805877109.
[I 2026-03-26 19:11:19,849] Trial 3 finished with value: 0.12328183961681857 and parameters: {'alpha': 327.79879319625803}. Best is trial 0 with value: 0.11137198805877109.
[I 2026-03-26 19:11:19,889] Trial 4 finished with value: 0.11180729306159898 and parameters: {'alpha': 3.191040478835297}. Best is trial 0 with value: 0.11137198805877109.
[I 2026-03-26 19:11:19,

Лучший RMSE Ridge: 0.1111
Лучшие параметры: {'alpha': 9.931760994296951}


In [17]:
lasso = Lasso(alpha=0.0004672260332236319, max_iter=10000)
ridge = Ridge(alpha=9.931760994296951)
lgbm  = lgb.LGBMRegressor(
    n_estimators=1812, learning_rate=0.01185060865103241,
    num_leaves=14, min_child_samples=5,
    subsample=0.9927865856597863, colsample_bytree=0.7009144370133,
    random_state=42, verbose=-1
)

voting_tuned = VotingRegressor([
    ('Lasso',    lasso),
    ('Ridge',    ridge),
    ('LightGBM', lgbm),
])

cv_scores = cross_val_score(voting_tuned, X_train, y,
                            cv=cv, scoring='neg_root_mean_squared_error')
print(f"RMSE Voting (натюненный): {-cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

RMSE Voting (натюненный): 0.1082 ± 0.0064
